# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadUsmanSaboor/Fly-Rank-Internship26/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

AI Referral Opportunity Scoring — *Freestyle direction*

I picked this lane because AI-referred traffic is becoming a distinct channel with its own discovery mechanics (AI overviews, chat-based search, agentic browsing). Unlike traditional SEO where we optimize for ranking position, AI referrals depend on whether a page is *selected* by an LLM as a source a noisier, sparser signal. This makes it a genuine machine-learning problem, not a dashboard exercise.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load packages and the starter dataset
import pandas as pd
import numpy as np

# Replace with your actual starter data path
df = pd.read_csv('content_refresh_anonymized.csv')

# Quick sanity check
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['observation_date'].min()} to {df['observation_date'].max()}")
print(f"Missing values: {df.isnull().sum().sum()}")

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### The search question
> *Which pages that currently receive zero (or near-zero) AI-referred traffic have the highest latent opportunity to attract AI referrals if optimized?*

### Unit of analysis
**One page** (URL path), observed at a single point in time with a 90-day trailing window of session data.

### The output
A ranked score (0–100) for every page in the corpus, with:
- **Score ≥ 70:** High opportunity — prioritize for AI-referral optimization (structured data, title/meta, content depth, FAQ schema)
- **Score 40–69:** Medium opportunity — monitor and light-touch optimize
- **Score < 40:** Low opportunity — deprioritize; do not allocate content-team cycles

### The decision someone makes
The **content operations lead** decides which pages go into the weekly AI-optimization sprint. Currently this is likely done by gut feel or total-traffic ranking. The model replaces that with an evidence-backed queue.

### The action someone takes
For high-opportunity pages, the content team:
1. Adds or improves structured data (FAQ, HowTo, Article schema)
2. Tunes titles and meta descriptions for AI-citation clarity (direct answers, entity-rich)
3. Expands thin content to match the depth of pages that already earn AI referrals

### The cost of a wrong recommendation
| Error type | Cost |
|---|---|
| **False positive** (score a page high, it never gets AI traffic after optimization) | Wasted content-team hours (~4–6 hrs/page × sprint capacity of 10 pages/week = 40–60 hrs/week). At $75/hr blended rate, that's **$3,000–$4,500/week** of misallocated effort. |
| **False negative** (miss a page that would have responded well to optimization) | Lost AI-referred sessions. If the average AI-referred session has higher engagement (longer dwell, lower bounce), the LTV loss compounds. For a page with 50k monthly sessions, even a 1% AI-referral lift is **500 sessions/month** left on the table. |
| **Systematic bias** (model favors one content type unfairly) | Team optimizes the wrong archetype for weeks, missing the true opportunity segment. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the decision-maker's current world vs. the model world
print("=" * 60)
print("CURRENT STATE vs. MODEL STATE")
print("=" * 60)

# Current: likely ranking by total traffic
top_by_traffic = df.nlargest(10, 'total_sessions_90d')
current_ai_sessions = top_by_traffic['ai_sessions_90d'].sum()
current_already_have_ai = (top_by_traffic['ai_sessions_90d'] > 0).sum()

print(f"\nCurrent heuristic (top 10 by total traffic):")
print(f"  Already have AI traffic: {current_already_have_ai}/10 pages")
print(f"  Combined AI sessions:    {current_ai_sessions:,}")
print(f"  -> {10 - current_already_have_ai} pages are already winning; optimizing them is diminishing returns.")

# Model world: find high-total-traffic pages with ZERO AI
zero_ai = df[df['ai_sessions_90d'] == 0]
top_zero_ai = zero_ai.nlargest(10, 'total_sessions_90d')
model_potential = top_zero_ai['total_sessions_90d'].sum()

print(f"\nModel target (top 10 zero-AI pages by total traffic):")
print(f"  Pages with zero AI:      10/10")
print(f"  Combined total sessions: {model_potential:,}")
print(f"  -> Every page is a greenfield opportunity.")

print(f"\n-> The gap between 'already winning' and 'should be winning' is the decision.")

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------------
# NUMBER 1: Sparsity — the core ML challenge
# --------------------------------------------------------
ai_positive = (df['ai_sessions_90d'] > 0).sum()
ai_positive_pct = ai_positive / len(df) * 100

print("=" * 55)
print("NUMBER 1: SPARSITY")
print("=" * 55)
print(f"Pages with AI referral traffic:     {ai_positive:,} / {len(df):,}")
print(f"Percentage with AI traffic:         {ai_positive_pct:.1f}%")
print(f"Percentage with ZERO AI traffic:    {100 - ai_positive_pct:.1f}%")
print(f"\n-> This is a classic imbalanced-learning problem.")
print(f"   A naive 'rank by total traffic' approach would ignore")
print(f"   the specific signal that predicts AI referral.")


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I can claim (with evidence)
- **Sparsity is real:** Only ~8.6% of pages in the starter dataset show any AI-referred sessions. This is not a rounding error; it is the defining characteristic of the problem.
- **Opportunity exists:** Hundreds of high-traffic, structured-data-enabled pages receive zero AI referrals. The gap is measurable in millions of sessions.
- **Content type matters:** FAQ and guide pages show higher AI penetration than product pages, but the absolute number of underperforming product pages is large. A model can learn cross-type patterns.
- **Ranking is the right output:** Because the action is a weekly sprint with limited capacity, we need a ranked queue, not just a binary classifier.

### What I cannot claim (yet)
- **Causality:** I am observing correlation between page features and AI referral. I cannot claim that adding structured data *causes* AI referral — only that it is associated with it in the observed data.
- **Generalization across time:** The 90-day window is a snapshot. AI referral patterns may shift as LLM behavior evolves. I need a time-based validation strategy (e.g., train on Q1, validate on Q2) before claiming the model is stable.
- **Action efficacy:** I have not measured whether optimizing a high-opportunity page actually increases its AI referral rate. That requires a controlled experiment or at least a pre/post holdout analysis.
- **Feature completeness:** The starter dataset has basic page features (content type, word count, structured data flag, title length, search position). I do not yet know if these are the *right* features — semantic quality, entity density, or backlink profile may matter more.
- **Business value:** I have estimated potential sessions at 2% conversion, but I do not yet know the conversion rate achievable by the content team, nor the revenue per AI-referred session.

### What would change my mind
- If a simple heuristic (e.g., "all FAQ pages with &gt;5k sessions") captures 90% of the opportunity, I would downgrade this from an ML project to a rules-based automation.
- If the AI-referral signal is entirely explained by total traffic (i.e., AI sessions proportional to total sessions with R² &gt; 0.9), then ranking by total traffic is sufficient and ML adds no value.
- If the content team cannot act on the ranked queue (no sprint capacity, no structured-data tooling), the model is useless regardless of its accuracy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.